# 03 · Clean and merge

Reads only files from `data_raw/` (no API calls), so it can be re-run from top to bottom in seconds.

**Outputs → `data_clean/`**

| File | Grain | Rows |
|---|---|---|
| `municipality_panel_2020_2024.csv` | municipality × year | 1,450 |
| `analyst_ads_final.csv` | one job ad | 6,801 |
| `dim_municipality.csv` | one municipality | 290 |

Every decision below is documented in `decisions.md`.

In [ ]:
import json

import pandas as pd

YEAR_FROM, YEAR_TO = 2020, 2024   # unemployment KPI only exists for these years
KEYS = ["municipality", "year"]

## 1. Load raw files
Municipality codes are always read as **text**: as numbers, `0114` would become `114` and every join would fail.

In [ ]:
pop_raw = pd.read_csv("../data_raw/kolada_population_raw.csv", dtype={"municipality": str})
unemp_raw = pd.read_csv("../data_raw/kolada_unemployment_raw.csv", dtype={"municipality": str})
rent_raw = pd.read_csv("../data_raw/scb_rent_raw.csv", dtype={"Region": str}, na_values=[".."])
income_raw = pd.read_csv("../data_raw/scb_income_raw.csv", dtype={"Region": str}, na_values=[".."])

for name, df in [("population", pop_raw), ("unemployment", unemp_raw), ("rent", rent_raw), ("income", income_raw)]:
    print(f"{name:13} {df.shape}")

## 2. Kolada: keep municipalities, totals, analysis years

Kolada returns 3,656 codes, not 290:
- `0000` = Sweden, `0001`–`0025` = regions → start with `00`
- `G…` = 3,344 municipality groups (unweighted averages of members)

These are aggregates of the municipality rows; keeping them would double count.

In [ ]:
def clean_kolada(df_raw: pd.DataFrame, value_name: str) -> pd.DataFrame:
    df = df_raw[
        (df_raw["gender"] == "T")                               # total, not women/men
        & (~df_raw["municipality"].str.startswith("G"))          # drop municipality groups
        & (~df_raw["municipality"].str.startswith("00"))         # drop Sweden + regions
        & (df_raw["period"].between(YEAR_FROM, YEAR_TO))
    ]
    return (
        df[["municipality", "period", "value"]]
        .rename(columns={"period": "year", "value": value_name})
        .reset_index(drop=True)
    )


pop = clean_kolada(pop_raw, "population")
unemp = clean_kolada(unemp_raw, "unemployment_rate")

## 3. SCB: wide → long

SCB returns one column per year (`HE0110J8 2020`, …). `melt` turns columns into rows (the pandas version of Power Query's *Unpivot columns*), then the year is taken from the last 4 characters.

In [ ]:
def melt_scb(df_raw: pd.DataFrame, code: str, value_name: str) -> pd.DataFrame:
    df = df_raw.melt(
        id_vars=["Region"],
        value_vars=[c for c in df_raw.columns if c.startswith(code)],
        var_name="column",
        value_name=value_name,
    )
    df["year"] = df["column"].str[-4:].astype("int64")   # explicit int64: same type on every OS
    return df.rename(columns={"Region": "municipality"})[["municipality", "year", value_name]]


income = melt_scb(income_raw, "HE0110J8", "median_income_tkr")
rent = melt_scb(rent_raw, "000000J4", "median_rent_sqm")
rent_moe = melt_scb(rent_raw, "000000J3", "rent_moe")

### Quality check before merging: 1,450 rows each, no duplicate keys

In [ ]:
tables = {"population": pop, "unemployment": unemp, "income": income, "rent": rent, "rent_moe": rent_moe}

for name, df in tables.items():
    print(
        f"{name:13} rows={len(df):,}",
        f"duplicates={df.duplicated(KEYS).sum()}",
        f"missing={df.iloc[:, 2].isna().sum()}",     # rent: 6 unpublished values (kept as missing)
    )

## 4. Merge into a municipality × year panel

Population is the base table (complete). Left joins keep every municipality even if another source has a gap.
`validate="one_to_one"` makes pandas raise an error if any key is duplicated, instead of silently multiplying rows.

In [ ]:
panel = (
    pop
    .merge(unemp, on=KEYS, how="left", validate="one_to_one")
    .merge(income, on=KEYS, how="left", validate="one_to_one")
    .merge(rent, on=KEYS, how="left", validate="one_to_one")
    .merge(rent_moe, on=KEYS, how="left", validate="one_to_one")
)
panel["population"] = panel["population"].astype("int64")

print(panel.shape)            # (1450, 7)
panel[panel["municipality"] == "1480"]   # spot check: Göteborg

## 5. Clean job ads

Starts from the raw JSON so every step is visible. A row count is printed after each rule (the *cleaning trail*).

In [ ]:
with open("../data_raw/jobtech_analyst_ads_2020_2024.json", encoding="utf-8") as f:
    ads_all = pd.json_normalize(json.load(f))

COLUMNS = {
    "id": "ad_id",
    "publication_date": "publication_date",
    "headline": "headline",
    "occupation.label": "occupation",
    "employer.name": "employer",
    "workplace_address.country": "country",
    "workplace_address.municipality_code": "municipality",
    "workplace_address.municipality": "municipality_name",
}
ads = ads_all[list(COLUMNS)].rename(columns=COLUMNS)
print("1. raw:                ", len(ads))

# Rule 1 – exact duplicates: same ad, different ID
ads = ads.drop_duplicates(subset=["headline", "employer", "municipality", "publication_date"], keep="first")
print("2. after duplicates:   ", len(ads))

# Rule 2 – keep Sweden. Sweden appears as both "Sverige" and "SE"; unknown country is kept
in_sweden = ads["country"].isin(["Sverige", "SE"]) | ads["country"].isna()
ads = ads[in_sweden].copy()
ads["country"] = ads["country"].replace({"SE": "Sverige"})
print("3. after abroad:       ", len(ads))

# Rule 3 – operational call-centre roles misclassified as business analysts.
# A broader keyword rule ("Workforce", "WFM") was tested and rejected: it also caught real BI roles.
wfm = ads["headline"].str.contains("Real Time Manager|Workforce Specialist", case=False, na=False)
ads = ads[~wfm].copy()
print("4. after WFM roles:    ", len(ads))

ads["year"] = pd.to_datetime(ads["publication_date"]).dt.year

## 6. Count ads per municipality and year, add to panel

Ads without a workplace municipality (~5%, often remote) cannot be placed on a municipality and are left out of the count, but stay in `analyst_ads_final.csv`.

**Zero vs missing:** after the join, a municipality-year with no ads is *empty*. Here empty means **zero** (the ad data is complete), so it is filled with 0.
Missing rent is *not* filled: there, empty means "unknown".

In [ ]:
ad_counts = (
    ads.dropna(subset=["municipality"])
    .groupby(KEYS)
    .size()
    .reset_index(name="analyst_ads")
)

panel = panel.merge(ad_counts, on=KEYS, how="left", validate="one_to_one")
panel["analyst_ads"] = panel["analyst_ads"].fillna(0).astype("int64")

# normalise by population so large and small municipalities can be compared
panel["ads_per_10k"] = panel["analyst_ads"] / panel["population"] * 10_000

print("ads in panel:", panel["analyst_ads"].sum())   # 6,501 = all ads with a known municipality
panel.sort_values("ads_per_10k", ascending=False).head(10)

## 7. Municipality dimension (name + region)

In [ ]:
muni_raw = pd.read_csv("../data_raw/kolada_municipalities_raw.csv", dtype={"id": str})

regions = (
    muni_raw[muni_raw["type"] == "L"][["id", "title"]]
    .rename(columns={"id": "region_id", "title": "region_name"})
)

dim_muni = (
    muni_raw[muni_raw["type"] == "K"][["id", "title"]]
    .rename(columns={"id": "municipality", "title": "municipality_name"})
)
# first two digits of a municipality code = its region, e.g. 1480 -> region 0014
dim_muni["region_id"] = "00" + dim_muni["municipality"].str[:2]
dim_muni = dim_muni.merge(regions, on="region_id", how="left", validate="many_to_one").reset_index(drop=True)

# referential integrity: every code in the panel exists in the dimension, and vice versa
assert set(panel["municipality"]) == set(dim_muni["municipality"])
print(dim_muni.shape, "| missing region names:", dim_muni["region_name"].isna().sum())

## 8. Save

In [ ]:
panel.to_csv("../data_clean/municipality_panel_2020_2024.csv", index=False)
ads.to_csv("../data_clean/analyst_ads_final.csv", index=False)
dim_muni.to_csv("../data_clean/dim_municipality.csv", index=False)

print("panel:", panel.shape, "| ads:", ads.shape, "| dim:", dim_muni.shape)